In [18]:
import pandas as pd
import matplotlib.pyplot as plt

In [11]:
df = pd.read_json('non_benchmark_961.json')
df_benchmark_summary = pd.read_csv('benchmark_summary.csv')
df_cat_delta_detail = pd.read_csv('category_delta_detail.csv')
df_cat_delta_summary = pd.read_csv('category_delta_summary.csv')

In [6]:
sample_df = df.groupby('category', group_keys=False).sample(n=10, random_state=42)
sample_df.to_csv('first_10_sample.csv')

In [14]:
df_benchmark_summary

,qid,own_category,segment,compared_category,similarity,js_divergence,religion_similarity,baseline_similarity,religion_delta
0,929,Applied Ethics & Moral Dilemmas,thought,"Law, Politics, Government & History",0.380626,0.619374,0.381504,0.362259,0.019245
1,929,Applied Ethics & Moral Dilemmas,thought,Technology & Definitions,0.373086,0.626914,0.381504,0.362259,0.019245
2,929,Applied Ethics & Moral Dilemmas,thought,Religion & Theology,0.381504,0.618496,0.381504,0.362259,0.019245
3,929,Applied Ethics & Moral Dilemmas,thought,Applied Ethics & Moral Dilemmas,0.446905,0.553095,0.381504,0.362259,0.019245
4,929,Applied Ethics & Moral Dilemmas,thought,Marriage & Romantic Partnerships,0.501071,0.498929,0.381504,0.362259,0.019245
...,...,...,...,...,...,...,...,...,...
195,845,Marriage & Romantic Partnerships,answer,Human Nature & Philosophy,0.346213,0.653787,0.311537,0.312670,-0.001133
196,845,Marriage & Romantic Partnerships,answer,"Science, Physics & Math",0.299042,0.700958,0.311537,0.312670,-0.001133
197,845,Marriage & Romantic Partnerships,answer,"Depression, Addiction & Feeling Lost",0.524853,0.475147,0.311537,0.312670,-0.001133
198,845,Marriage & Romantic Partnerships,answer,Family Duty & Obligations,0.497218,0.502782,0.311537,0.312670,-0.001133


In [12]:
df_cat_delta_summary

,compared_category,segment,mean,std,count
0,Marriage & Romantic Partnerships,answer,0.087880,0.070745,10
1,Family Duty & Obligations,answer,0.079790,0.065167,10
2,"Depression, Addiction & Feeling Lost",answer,0.069018,0.091330,10
3,"Grief, Loss & Death",answer,0.046393,0.064981,10
4,Applied Ethics & Moral Dilemmas,answer,0.045826,0.050611,10
5,Human Nature & Philosophy,answer,0.039615,0.041957,10
6,Religion & Theology,answer,0.006807,0.027026,10
7,"Science, Physics & Math",answer,0.005648,0.015351,10
8,Technology & Definitions,answer,0.004201,0.011504,10
9,"Law, Politics, Government & History",answer,-0.009849,0.006110,10


In [13]:
df_cat_delta_detail

,qid,own_category,segment,compared_category,similarity,baseline_similarity,category_delta
0,776,Family Duty & Obligations,answer,Applied Ethics & Moral Dilemmas,0.417582,0.299893,0.117689
1,801,Inner Peace & Personal Growth,answer,Applied Ethics & Moral Dilemmas,0.345626,0.286691,0.058935
2,804,Everyday Ethics & Honesty,answer,Applied Ethics & Moral Dilemmas,0.316473,0.320955,-0.004481
3,840,"Depression, Addiction & Feeling Lost",answer,Applied Ethics & Moral Dilemmas,0.319185,0.316375,0.002811
4,845,Marriage & Romantic Partnerships,answer,Applied Ethics & Moral Dilemmas,0.383516,0.312670,0.070846
...,...,...,...,...,...,...,...
195,846,Human Nature & Philosophy,thought,Technology & Definitions,0.324105,0.331364,-0.007259
196,929,Applied Ethics & Moral Dilemmas,thought,Technology & Definitions,0.373086,0.362259,0.010827
197,995,Friendship & Trust,thought,Technology & Definitions,0.306773,0.306855,-0.000082
198,998,Cosmology & Speculative Science,thought,Technology & Definitions,0.284908,0.291159,-0.006251


In [20]:
SURFACE = "#fcfcfb"
TEXT_PRIMARY = "#0b0b0b"
TEXT_SECONDARY = "#52514e"
GRIDLINE = "#e1e0d9"
BASELINE_LINE_COLOR = "#9c9b93"

# Fresh 10-color qualitative palette (matplotlib tab10) -- the project's
# CATEGORY_COLORS reuses 2 hues (Family Duty/Grief), which is fine when
# they're folded into "Other" elsewhere but would clash here since all 10
# categories are on screen simultaneously.
CATEGORY_COLORS = {
    "Religion & Theology":                  "#1f77b4",
    "Law, Politics, Government & History":  "#ff7f0e",
    "Technology & Definitions":              "#2ca02c",
    "Applied Ethics & Moral Dilemmas":       "#d62728",
    "Marriage & Romantic Partnerships":      "#9467bd",
    "Human Nature & Philosophy":             "#8c564b",
    "Science, Physics & Math":               "#e377c2",
    "Depression, Addiction & Feeling Lost":  "#7f7f7f",
    "Family Duty & Obligations":             "#bcbd22",
    "Grief, Loss & Death":                   "#17becf",
}

df = pd.read_csv("category_delta_summary.csv")
piv = df.pivot(index="compared_category", columns="segment", values="mean")

# Average RAW baseline similarity (not delta -- baseline_similarity is the
# thing category_delta already subtracts out) by segment, from the
# per-question benchmark summary. This is on a much larger absolute scale
# (~0.30) than the deltas (~-0.01 to 0.10), since it's raw similarity
# rather than baseline-calibrated signal -- so it gets its own right-hand
# axis rather than squashing the delta lines onto the bottom of the plot.
baseline_avg = (
    df_benchmark_summary.drop_duplicates(subset=["qid", "segment"])
    .groupby("segment")["baseline_similarity"].mean()
)

fig, ax = plt.subplots(figsize=(7, 6.5), facecolor=SURFACE)
ax.set_facecolor(SURFACE)

x = [0, 1]
for category, row in piv.iterrows():
    color = CATEGORY_COLORS[category]
    y = [row["thought"], row["answer"]]
    ax.plot(x, y, color=color, linewidth=2, marker="o", markersize=7,
             label=category, zorder=2)

ax.axhline(0, color=TEXT_SECONDARY, linewidth=1, linestyle="--", zorder=0)
ax.set_xticks(x)
ax.set_xticklabels(["Thought", "Answer"], fontsize=10, color=TEXT_PRIMARY)
ax.set_xlim(-0.2, 1.2)
ax.set_ylabel("Category delta  (similarity to category − avg. similarity to baseline categories)",
              color=TEXT_SECONDARY, fontsize=9)
ax.set_title("Category delta: thought → answer (benchmark pilot, n=10)",
             color=TEXT_PRIMARY, fontsize=11.5, loc="left")
ax.grid(True, color=GRIDLINE, linewidth=0.8, axis="y", zorder=0)
ax.spines[["top", "right"]].set_visible(False)
ax.spines[["left", "bottom"]].set_color(GRIDLINE)
ax.tick_params(colors=TEXT_SECONDARY, labelsize=8)

# Secondary axis: average raw baseline similarity, thought vs answer.
ax2 = ax.twinx()
ax2.plot(x, [baseline_avg["thought"], baseline_avg["answer"]],
          color=BASELINE_LINE_COLOR, linewidth=2, linestyle="--", marker="s",
          markersize=6, zorder=2, label="Avg. baseline similarity (raw, right axis)")
ax2.set_ylabel("Avg. baseline similarity (raw)", color=TEXT_SECONDARY, fontsize=9)
ax2.spines[["top", "left"]].set_visible(False)
ax2.spines["right"].set_color(GRIDLINE)
ax2.tick_params(colors=TEXT_SECONDARY, labelsize=8)

lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, frameon=False, fontsize=8,
          labelcolor=TEXT_PRIMARY, loc="center left", bbox_to_anchor=(1.15, 0.5))

fig.tight_layout()
fig.savefig("category_delta_slope.png", dpi=150, facecolor=SURFACE, bbox_inches="tight")
plt.close(fig)
print("Wrote category_delta_slope.png")
print(baseline_avg)

Wrote category_delta_slope.png
segment
answer     0.308910
thought    0.324201
Name: baseline_similarity, dtype: float64
